#

In [12]:
ip_port = "132.72.66.154:9982"

In [13]:
url = f"http://{ip_port}/api/generate"

In [14]:
import json
import pandas as pd
import requests

def extract_full_text(response_text):
    """
    This function extracts and concatenates text from the 'response' field in each JSON line of the response.
    
    Parameters:
    response_text (str): The raw response text, with each line being a JSON string.

    Returns:
    str: The concatenated text from all 'response' fields.
    """
    full_text = ""  # Variable to store the complete text
    
    # Iterate over each line in the response
    for line in response_text.text.splitlines():
        try:
            data = json.loads(line)  # Parse each JSON line
            if "response" in data:  # Check if the "response" field exists
                full_text += data["response"]  # Append the text
        except json.JSONDecodeError:
            print("Unable to parse line:", line)
    
    return full_text

In [15]:

def check_relevance(abstract,title,inclusion_criteria,exclusion_criteria):
    prompt = (
        "You are a researcher rigorously screening titles and abstracts of scientific papers for inclusion or exclusion in a review paper. "
        "Use the criteria below to inform your decision. If any exclusion criteria are met or not all inclusion criteria are met, exclude the article. "
        "If all inclusion criteria are met, include the article. Only type 'included' or 'excluded' to indicate your decision. Do not type anything else.\n\n"
        f"Title: {title}\n"
        f"Abstract: {abstract}\n\n"
        f"Inclusion criteria: {inclusion_criteria}\n"
        f"Exclusion criteria: {exclusion_criteria}\n\n"
        "Decision: {here}"
        "Explain: {here}"
    )
    
    payload = {
        "model": "phi4",
        "prompt": prompt,
        "temperature": 0.2,  
        "top_p": 0.2       
    }

    # Make a POST request
    response = requests.post(url, json=payload)

    # Validate the response
    if response.status_code == 200:
        # decision = response.json().get("choices", [{}])[0].get("text", "").strip()
        decision = extract_full_text(response)
        # print(decision)
        # print("------------------------------------------------")
        return decision #if decision in {"included", "excluded"} else decision
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

In [16]:
inclusion_criteria = (
    "1. The article discusses barriers or facilitators to nurses' involvement in research activities.\n"
    "2. The article identifies challenges nurses face in conducting research, such as educational deficiencies, time constraints, organizational or cultural barriers, or lack of resources.\n"
    "3. The article proposes solutions or strategies to overcome barriers to nursing research participation.\n"
    "4. The article addresses the integration of research findings into clinical practice (Evidence-Based Practice).\n"
    "5. The article examines organizational cultures, policies, or leadership styles affecting nurses’ research involvement.\n"

)

#    "6. The article explores the use of technology, such as NLP tools, to enhance nursing research or systematic reviews."

exclusion_criteria = (

    "1. The article does not focus on nursing research or its associated barriers and solutions.\n"
    "2. The article lacks sufficient detail to evaluate against the inclusion criteria."
)

#    "1. The article is not written in English.\n"
    # "1. The article is a review paper or meta-analysis.\n"


In [19]:
check_relevance(df.iloc[0].Abstract, df.iloc[0].Title, inclusion_criteria, exclusion_criteria)

"**Decision:** Excluded\n\n**Explanation:** The abstract of this study focuses on identifying elements of missed nursing care on medical-surgical units and reasons for missing this care as reported by nursing staff. It does not address barriers or facilitators to nurses' involvement in research activities, challenges faced by nurses in conducting research, solutions to overcome these barriers, integration of research findings into clinical practice (Evidence-Based Practice), or the impact of organizational cultures, policies, or leadership styles on nursing research involvement. Therefore, it does not meet any of the inclusion criteria specified and should be excluded based on exclusion criterion 1, as it does not focus on nursing research or its associated barriers and solutions."

In [20]:
df = pd.read_pickle('combined_df_all_databases_only_english.pkl')


In [21]:
from tqdm import tqdm

# Define a function to process each row and update the CSV incrementally
def add_decision_column_and_save(df, inclusion_criteria, exclusion_criteria, output_file):
    # Check if the file already exists
    import os
    if os.path.exists(output_file):
        # If the file exists, load it
        processed_df = pd.read_csv(output_file)
        start_index = len(processed_df)  # Start from where we left off
    else:
        # If the file doesn't exist, initialize an empty DataFrame
        processed_df = pd.DataFrame(columns=df.columns.tolist() + ['Decision'])
        start_index = 0

    # Iterate over rows with tqdm for progress tracking, starting from the last processed row
    for i in tqdm(range(start_index, len(df)), desc="Processing rows"):
        # Extract abstract and title for each row
        abstract = df.iloc[i].Abstract
        title = df.iloc[i].Title
        
        # Call the check_relevance function
        decision = check_relevance(abstract, title, inclusion_criteria, exclusion_criteria)
        
        # Add the current row and decision to the processed DataFrame
        current_row = df.iloc[i].to_dict()
        current_row['Decision'] = decision
        processed_df = pd.concat([processed_df, pd.DataFrame([current_row])], ignore_index=True)
        
        # Save the updated DataFrame to the CSV file after each iteration
        processed_df.to_csv(output_file, index=False)

    return processed_df


# File to save the updated DataFrame
output_file = "updated_decisions_incremental.csv"

# Call the function with your DataFrame
df_with_decisions = add_decision_column_and_save(df, inclusion_criteria, exclusion_criteria, output_file)

# Display the first few rows of the final DataFrame
print(df_with_decisions.head())


Processing rows: 100%|██████████| 5990/5990 [7:06:37<00:00,  4.27s/it]  

    PubMed ID                               DOI  \
0  16985399.0  10.1097/00001786-200610000-00006   
1  15946170.0  10.1111/j.1365-2934.2005.00541.x   
2  17147757.0  10.1111/j.1741-6787.2004.04008.x   
3  19076465.0  10.1111/j.1744-6198.2008.00115.x   
4  15946274.0  10.1111/j.1365-2702.2005.01136.x   

                                               Title  Year  \
0           Missed nursing care: a qualitative study  2006   
1                              Confirming mentorship  2005   
2                     Methods in translation science  2004   
3                     Competency: a concept analysis  2008   
4  Clinical education facilitators: a literature ...  2005   

                                            Abstract Author Keywords english  \
0  The purpose of this study was to determine nur...              []      en   
1  Mentorship is related to nurses' success in nu...              []      en   
2  The gap between research findings and practice...              []      en   


In [25]:
# import pandas as pd

# df= pd.read_csv('updated_decisions_incremental.csv')
# df

In [24]:
df_with_decisions.shape

(17208, 8)

In [26]:
df_with_decisions

,PubMed ID,DOI,Title,Year,Abstract,Author Keywords,english,Decision
0,16985399.0,10.1097/00001786-200610000-00006,Missed nursing care: a qualitative study,2006,The purpose of this study was to determine nur...,[],en,Excluded\n\nExplain: The title and abstract of...
1,15946170.0,10.1111/j.1365-2934.2005.00541.x,Confirming mentorship,2005,Mentorship is related to nurses' success in nu...,[],en,**Decision:** Excluded\n\n**Explanation:** \n\...
2,17147757.0,10.1111/j.1741-6787.2004.04008.x,Methods in translation science,2004,The gap between research findings and practice...,[],en,**Decision:** Excluded\n\n**Explanation:** \nT...
3,19076465.0,10.1111/j.1744-6198.2008.00115.x,Competency: a concept analysis,2008,Concept analysis of competency.\nThe purpose o...,[],en,Decision: excluded\n\nExplain: The abstract in...
4,15946274.0,10.1111/j.1365-2702.2005.01136.x,Clinical education facilitators: a literature ...,2005,"The aim of this literature review, set within ...",[],en,Excluded\n\nExplain: The abstract of the paper...
...,...,...,...,...,...,...,...,...
17203,35332562.0,10.1111/jan.15219,What the curtains do not shield: A phenomenolo...,2022,Aims: To explore healthcare professionals' exp...,cardiac arrest; cardiopulmonary resuscitation;...,en,**Decision:** Excluded\n\n**Explain:** \nThe ...
17204,31666038.0,10.1186/s12904-019-0481-y,End-of-life and palliative care of patients on...,2019,Background: Despite complex illness trajectori...,End-of-life care; Palliative care; End-stage k...,en,**Decision:** Excluded\n\n**Explanation:** \n...
17205,38287332.0,10.1186/s12913-024-10617-9,The Quality in Acute Stroke Care (QASC) global...,2024,BackgroundVariation in hospital stroke care is...,Implementation science; Facilitation; Stroke; ...,en,**Decision:** Excluded\n\n**Explanation:** \n\...
17206,38877433.0,10.1186/s12877-024-05124-9,Exploring the impact of pharmacist-supported m...,2024,Background Dementia is a major global public h...,Clinical pharmacy; Collaboration; Dementia; Dr...,en,**Decision:** Excluded\n\n**Explanation:** The...


In [28]:
import re

# Define a function to extract "Inclusion" or "Exclusion" from the Decision column
def extract_decision(decision_text):
    decision_text = str(decision_text)
    match = re.search(r'\b(Inclusion|Exclusion|Included|Excluded)\b', decision_text, re.IGNORECASE)
    if match:
        return match.group(0).capitalize()  # Capitalize to maintain consistency (e.g., "Excluded")
    return "Unknown"  # Return "Unknown" if no match is found

# Apply the function to the Decision column
df_with_decisions['Decision Type'] = df_with_decisions['Decision'].apply(extract_decision)

# Display the DataFrame with the extracted decision type
print(df_with_decisions[['Decision', 'Decision Type']].head())


                                            Decision Decision Type
0  Excluded\n\nExplain: The title and abstract of...      Excluded
1  **Decision:** Excluded\n\n**Explanation:** \n\...      Excluded
2  **Decision:** Excluded\n\n**Explanation:** \nT...      Excluded
3  Decision: excluded\n\nExplain: The abstract in...      Excluded
4  Excluded\n\nExplain: The abstract of the paper...      Excluded


In [30]:
df_with_decisions['Decision Type'].value_counts()


Decision Type
Excluded     16485
Included       718
Inclusion        4
Unknown          1
Name: count, dtype: int64

In [33]:
df_with_decisions[df_with_decisions['Decision Type'] == 'Excluded'].sample(100).to_csv('100_Excluded.csv')